In [16]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import sklearn
import sklearn.datasets
import sklearn.linear_model
%matplotlib inline

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
def sigmoid(x):
    s = 1 / (np.exp(-x) + 1)
    return s

In [18]:
def layer_sizes(X, Y):
    n_x = X.shape[0]
    n_h = 4
    n_y = Y.shape[0]
    return (n_x, n_h, n_y)

In [19]:
def initialize_parameters(n_x, n_h, n_y):
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    parameters = {
        "W1" : W1,
        "b1" : b1,
        "W2" : W2,
        "b2" : b2
    }
    return parameters

In [20]:
def forward_propagation(X, parameters):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = np.tanh(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {
        "Z1" : Z1,
        "A1" : A1,
        "Z2" : Z2,
        "A2" : A2
    }
    return cache

In [ ]:
def compute_cost(A2, Y):
    m = Y.shape[1]
    logprobs = np.multiply(np.log(A2), Y) + np.multiply((1 - Y), np.log(1 - A2))
    cost = - np.sum(logprobs) / m
    cost = float(np.squeeze(cost))
    return cost 

In [22]:
def backward_propagation(parameters, cache, X, Y):
    m = X.shape[1]

    W1 = parameters["W1"]
    W2 = parameters["W2"]

    A1 = cache["A1"]
    A2 = cache["A2"]

    dZ2 = A2 - Y
    dW2 = np.dot(dZ2, A1.T) / m
    db2 = np.sum(dZ2, axis = 1, keepdims = True) / m
    dZ1 = np.dot(W2.T, dZ2) * (1 - np.power(A1, 2))
    dW1 = np.dot(dZ1, X.T) / m 
    db1 = np.sum(dZ1, axis = 1, keepdims = True) / m

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }
    
    return grads

In [23]:
def update_parameters(parameters, grads, learning_rate = 1.2):
    W1 = copy.deepcopy(parameters["W1"])
    b1 = parameters["b1"]
    W2 = copy.deepcopy(parameters["W2"])
    b2 = parameters["b2"]

    dW1 = grads["dW1"]
    db1 = grads["db1"]
    dW2 = grads["dW2"]
    db2 = grads["db2"]

    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    parameters = {
            "W1": W1,
            "b1": b1,
            "W2": W2,
            "b2": b2
        }
    
    return parameters

In [24]:
def nn_model(X, Y, n_h, num_iterations = 10000, print_cost = False):

    np.random.seed(3)
    n_x = layer_sizes(X, Y)[0]
    n_y = layer_sizes(X, Y)[2]

    parameters = initialize_parameters(n_x, n_h, n_y)

    for i in range(0, num_iterations):
        cache = forward_propagation(X, parameters)
        A2 = cache["A2"]
        cost = compute_cost(A2, Y)
        grads = backward_propagation(parameters, cache, X, Y)
        parameters = update_parameters(parameters, grads)

    if print_cost and i % 1000 == 0:
        print ("Cost after iteration %i: %f" %(i, cost))

    return parameters

In [25]:
def predict(parameters, X):

    cache = forward_propagation(X, parameters)
    A2 = cache["A2"]
    predictions = (A2 > 0.5)

    return predictions

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [27]:

path = r'C:\Users\Admin\Documents\Work and pain\Machine Learning\My_Model\Nerural_Network\datasets\heart_diseases\heart.csv'

df = pd.read_csv(path)

X_raw = df.drop('target', axis=1).values
Y_raw = df['target'].values
X_train, X_test, Y_train, Y_test = train_test_split(X_raw, Y_raw, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_nn = X_train_scaled.T
Y_train_nn = Y_train.reshape(1, -1)

X_test_nn = X_test_scaled.T
Y_test_nn = Y_test.reshape(1, -1)

print("X_train_nn's shape:", X_train_nn.shape) 
print("Y_train_nn's shape:", Y_train_nn.shape)

X_train_nn's shape: (13, 242)
Y_train_nn's shape: (1, 242)


In [28]:
parameters = nn_model(X_train_nn, Y_train_nn, n_h=4, num_iterations=10000, print_cost=True)
# parameters

In [29]:
predictions = predict(parameters, X_train_nn)
print ('Accuracy: %d' % float((np.dot(Y_train_nn, predictions.T) 
                               + np.dot(1 - Y_train_nn, 1 - predictions.T)) 
                               / float(Y_train_nn.size) * 100) + '%')

Accuracy: 97%


C:\Users\Admin\AppData\Local\Temp\ipykernel_21824\4059765872.py:2: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print ('Accuracy: %d' % float((np.dot(Y_train_nn, predictions.T)


In [30]:
test_predictions = predict(parameters, X_test_nn)
test_accuracy = np.mean(test_predictions == Y_test_nn) * 100
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 81.97%
